# Dermoscopy Copilot — 01. Data Preparation and Train/Validation/Test Split

**Course:** Human-Centered Artificial Intelligence (HCAI) · MSc in Biomedical Engineering · University of Coimbra
**Group 13:** Carolina Garrido, Sara Santos, Sara Seco

## Purpose of this notebook

This notebook prepares the data used throughout the project. Specifically, it:

1. Downloads the **HAM10000** dataset (dermoscopic images and metadata) and the corresponding **lesion segmentation masks**;
2. Performs an initial exploratory analysis (class distribution and number of images per lesion);
3. Links each `image_id` to its image file;
4. Splits the data into **training (~70%)**, **validation (~15%)** and **test (~15%)** sets, stratified by diagnosis and grouped by lesion to prevent data leakage;
5. Saves the splits as CSV files, to be reused by the remaining notebooks.

> **Note:** the split is performed only once, with a fixed seed (`random_state=42`), to ensure reproducibility.

## 1. Library imports

In [ ]:
import os
from glob import glob

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42  # fixed seed for reproducibility

## 2. Dataset download

Two datasets are used:

- **Skin Cancer MNIST: HAM10000** — 10,015 dermoscopic images and a metadata file (`HAM10000_metadata.csv`);
- **HAM10000 Lesion Segmentations**  — binary masks delimiting the lesion region in each image, required for the interpretability evaluation (comparison with Grad-CAM heatmaps).

Both are downloaded with the `kagglehub` library, which stores the files in a local cache.

In [ ]:
# Images + metadata
path_img = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Images:", path_img)

# Lesion segmentation masks 
path_seg = kagglehub.dataset_download("tschandl/ham10000-lesion-segmentations")
print("Masks:", path_seg)

# Inspect the folder structure
for p in [path_img, path_seg]:
    print("\n", p)
    for item in os.listdir(p):
        print("  ", item)

## 3. Exploratory analysis of the metadata

The metadata contains, for each image, the image identifier (`image_id`), the lesion identifier (`lesion_id`) and the diagnosis (`dx`). Two aspects are particularly relevant for the experimental design:

- **Class imbalance:** the number of images varies considerably across the seven diagnostic categories, which motivates the use of class weights and macro-averaged metrics (Balanced Accuracy, macro F1-score, etc.);
- **Multiple images per lesion:** a single lesion may have more than one image. If images of the same lesion end up in different splits, the model is evaluated on images nearly identical to those seen during training, which inflates performance estimates. The split is therefore performed **by lesion** rather than by image.

In [ ]:
df = pd.read_csv(os.path.join(path_img, "HAM10000_metadata.csv"))

print("Dataframe shape:", df.shape)  # expected: (10015, 7)
display(df.head())

print("\nNumber of images per diagnosis (class imbalance):")
print(df["dx"].value_counts())

print("\nNumber of lesions with N images (index = number of images per lesion):")
print(df["lesion_id"].value_counts().value_counts())

## 4. Linking each image to its file

The images are spread across several subfolders. A dictionary mapping each `image_id` to the path of its `.jpg` file is built, and a `path` column is added to the dataframe.

In [ ]:
# Dictionary {image_id: file path}
img_paths = {
    os.path.splitext(os.path.basename(f))[0]: f
    for f in glob(os.path.join(path_img, "*", "*.jpg"))
}

# New path column with the file path of each image
df["path"] = df["image_id"].map(img_paths)

# Sanity check: every image must have a file associated with it
print("Images without an associated file:", df["path"].isna().sum())  # expected: 0

## 5. Example images per class

Visual check that images were correctly linked to their diagnoses (one image per class).

In [ ]:
classes = df["dx"].unique()  # "dx" = diagnosis

fig, axes = plt.subplots(1, len(classes), figsize=(18, 3))
for ax, c in zip(axes, classes):
    row = df[df["dx"] == c].iloc[0]
    ax.imshow(Image.open(row["path"]))
    ax.set_title(c)
    ax.axis("off")
plt.show()

## 6. Train / validation / test split

The data is divided into three splits:

| Split | Purpose | Approx. proportion |
|---|---|---|
| **Training** | Fitting the model parameters | ~70% |
| **Validation** | Hyperparameter selection and model tuning | ~15% |
| **Test** | Final evaluation, used only at the end | ~15% |

`StratifiedGroupKFold` is used, which simultaneously ensures:

- **Stratification** by diagnosis (`dx`), keeping similar class proportions across splits;
- **Grouping** by `lesion_id`, ensuring that all images of the same lesion fall in the same split.

The split is performed in two steps: first the test set is separated (1 of 7 folds ≈ 15%) and then, from the remainder, the validation set is separated (1 of 6 folds ≈ 14% of the total).

In [ ]:
# Separate the test set (1 out of 7 folds ≈ 15%)
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(
    sgkf.split(df, df["dx"], groups=df["lesion_id"])
)
trainval, test = df.iloc[trainval_idx], df.iloc[test_idx]

# Separate the validation set from the remainder (1 out of 6 folds)
sgkf2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
train_idx, val_idx = next(
    sgkf2.split(trainval, trainval["dx"], groups=trainval["lesion_id"])
)
train, val = trainval.iloc[train_idx], trainval.iloc[val_idx]

## 7. Split verification

We confirm that:

- the size of each split is as expected;
- there is **no lesion overlap** between splits (no data leakage);
- all classes are represented in the test set.

In [ ]:
print("Number of images (train / val / test):", len(train), len(val), len(test))

# No lesion may be shared between splits (expected: set())
print("Lesions shared train/test:", set(train.lesion_id) & set(test.lesion_id))
print("Lesions shared train/val: ", set(train.lesion_id) & set(val.lesion_id))
print("Lesions shared val/test:  ", set(val.lesion_id) & set(test.lesion_id))

# All classes must be represented in the test set
print("\nClass distribution in the test set:")
print(test["dx"].value_counts())

## 8. Saving the splits

The splits are saved as CSV files (`train.csv`, `val.csv`, `test.csv`) in the `splits/` folder, containing the image identifier, the lesion identifier and the diagnosis. The remaining notebooks load these files, guaranteeing that all experiments use exactly the same split.

In [ ]:
os.makedirs("splits", exist_ok=True)

cols = ["image_id", "lesion_id", "dx"]
train[cols].to_csv("splits/train.csv", index=False)
val[cols].to_csv("splits/val.csv", index=False)
test[cols].to_csv("splits/test.csv", index=False)

print("Splits saved to 'splits/'.")